# Importacion de librerias

In [7]:
import pandas as pd
from dotenv import dotenv_values
import requests
from io import BytesIO
import unicodedata
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu


#ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [8]:
#Mostrar todas la columnas de los DataFrame
pd.set_option("display.max_columns", None)  # mostrar todas las columnas

# Utilizar los datos desde github

In [9]:
config = dotenv_values(dotenv_path=".env")

In [10]:
url = config['DATA_FILE_URL']

headers = {"Authorization": f"Bearer {config['GH_TOKEN']}"}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()  # Check if the request was successful

xlsx_data = pd.read_excel(BytesIO(r.content), header=None, sheet_name=None, engine="openpyxl")

# Funciones varias

In [11]:
# Limpiar cada hoja eliminando filas completamente vacías y restableciendo el índice
cleaned_sheets = {}
for name, sheet in xlsx_data.items():
    sheet = sheet.dropna(how="all")
    sheet = sheet.reset_index(drop=True)
    cleaned_sheets[name] = sheet

In [12]:
#Funcion para limpiar los nombres de las columnas y dejarlas mas estandarizadas
def clean_value(cadena):
  #Se eliminan espacios, se pasan a minusculas, se reemplazan espacios por guiones bajos, se eliminan acentos y caracteres especiales
  cadena = str(cadena).strip().lower().replace(' ', '_').replace('-','').replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u").replace(".","").replace("(","").replace(")","")
  return cadena

In [13]:
# funcion para detectar el formato de la tabla basándose en palabras clave en columnas específicas
def detect_table_format(sheet, keywords=['espacio'], check_columns=[0, 1]):
    """
    Detecta el formato de la tabla basándose en palabras clave en columnas específicas.
    
    Parámetros:
    - sheet: DataFrame a analizar
    - keywords: lista de palabras clave a buscar
    - check_columns: índices de columnas a verificar (0=primera columna, 1=segunda columna)
    
    Retorna:
    - 0 si la palabra clave se encuentra en la primera columna
    - 1 si la palabra clave se encuentra en la segunda columna
    - 'no determinada' si no se encuentra en ninguna
    """
    
    for col_idx in check_columns:
        if col_idx < len(sheet.columns):
            # Vectorizar la búsqueda: convierte a string y se limpia la columna de una sola vez
            col_data = sheet.iloc[:, col_idx].astype(str).str.lower().str.strip()
            
            # Aplicar transformaciones necesarias vectorizadas
            col_data_clean = col_data.str.replace(' ', '_', regex=False)
            col_data_clean = col_data_clean.str.replace('[áéíóú]', 
                lambda m: {'á':'a', 'é':'e', 'í':'i', 'ó':'o', 'ú':'u'}.get(m.group(0), m.group(0)), 
                regex=True)
            col_data_clean = col_data_clean.str.replace(r'[\-\.\(\)]', '', regex=True)
            
            # Verificar si alguna palabra clave existe en la columna
            pattern = '|'.join(keywords)
            if col_data_clean.str.contains(pattern, na=False, case=False).any():
                return col_idx
    
    return 'no determinada'

In [14]:
#Funcion para extraer la clasificación DC y edad del nombre de la hoja
def extract_patient_info(sheet_name):
    """Extrae la clasificación DC y edad del nombre de la hoja"""
    if 'F06' in str(sheet_name).upper():
        #Deterioro cognitivo leve
        dc = 1
    elif 'F02' in str(sheet_name).upper():
        #Demencia
        dc = 2
    elif 'GC' in str(sheet_name).upper():
        #Grupo de control
        dc = 0
    else: 
        dc = 'No determinada'
    
    age = str(sheet_name).split('-')[-1].strip()
    return dc, age

In [15]:
#funcion para extraer los valores de features de una tabla
def extract_features_from_table(sheet, headers_col, value_col, features, headers_clean=None):
    """
    Extrae los valores de features de una tabla.
    
    Parámetros:
    - sheet: DataFrame de la tabla
    - headers_col: índice de la columna con headers
    - value_col: índice de la columna con valores
    - features: lista de features a buscar
    - headers_clean: diccionario precominado de headers limpios
    
    Retorna: diccionario con los valores encontrados
    """
    # Pre-procesar headers para limpiarlos
    if headers_clean is None:
        headers = sheet.iloc[:, headers_col].astype(str)
        headers_clean = headers.apply(clean_value)
    else:
        headers_clean = headers_clean
    
    result = {}
    
    # Crear un diccionario para búsqueda rápida: feature_clean -> índice de fila
    feature_to_idx = {}
    for feature in features:
        mask = headers_clean.str.contains(feature, case=False, na=False)
        if mask.any():
            feature_to_idx[feature] = headers_clean[mask].index[0]
    
    # Extraer los valores para cada feature
    for feature in features:
        if feature in feature_to_idx:
            idx = feature_to_idx[feature]
            result[feature] = sheet.iloc[idx, value_col]
        else:
            result[feature] = None
    
    return result

In [16]:
# Función principal para buscar y extraer los valores de features de todas las tablas
def search_values(data, features, type_of_table):
    """
    Busca y extrae los valores de features de todas las tablas,
    diferenciando entre tabla formato 0 y formato 1.
    """
    # Configuración de parámetros para cada formato de tabla
    table_configs = {
        0: {'headers_col': 0, 'value_col': 3, 'nivel_estudio': 0},
        1: {'headers_col': 1, 'value_col': 4, 'nivel_estudio': 1}
    }
    
    results = []
    
    # Procesar todas las hojas
    for sheet_name, sheet in data.items():
        table_format = type_of_table[sheet_name]
        
        # Ignorar si el formato no se pudo determinar
        if table_format == 'no determinada':
            continue
        
        config = table_configs[table_format]
        
        # Extraer información del paciente
        dc, age = extract_patient_info(sheet_name)
        
        # Pre-procesar headers una sola vez para esta tabla
        headers = sheet.iloc[:, config['headers_col']].astype(str)
        headers_clean = headers.apply(clean_value)
        
        # Extraer features
        features_dict = extract_features_from_table(
            sheet, 
            config['headers_col'], 
            config['value_col'], 
            features,
            headers_clean
        )
        
        # Construir registro del paciente
        patient_data = {
            'sheet_name': sheet_name,
            'nivel_estudio': config['nivel_estudio'],
            'dc': dc,
            'age': age,
            **features_dict
        }
        
        results.append(patient_data)
    
    # Convertir a DataFrame
    dataset_final = pd.DataFrame(results)
    return dataset_final

# Lectura de datos

### Mapeo de columnas para los dos formatos de tablas

In [17]:
features = [
    
   #Tabla formato 0 -> Escolaridad baja
   #Tabla formato 1 -> Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 0
  'atencion_sostenida_auditiva', # -> 'digitos_en_progresion',
  'atencion_sostenida_visual', #Sin equivalencia
  'atencion_selectiva_visual', #Sin equivalencia
  'atencion_dividida_visual', # -> deteccion_visual
  #tabla 1
  'digitos_en_progresion',
#   'cubos_progresion',
  'deteccion_visual',
#   'deteccion_digitos_total',
  'series_sucesivas',

  #####Lenguaje
  
  #tabla 0
  'denominacion',
  'comprension_de_ordenes', #No crucial
  'material_verbal_complejo', #No crucial
  
  #tabla 1
  # 'denominacion'
  'semejanzas',
  # 'material_verbal_complejo',
  'comprension_de_ordenes',

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #verbal del rey 
  
  #Tabla 0 - california verbal learning test
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',
  #tabla 0 - aprendizaje verbal del rey
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',
  #Tabla 1 #No tiene equivalencia con los otros test de la tabla 0
  'curva_de_memoria_volumen_promedio', 
  'memoria_verbal_espontanea_total',
  'memoria_verbal_claves_total',
  'memoria_verbal_reconocimiento',
  'memoria_lógica_promedio_historias',
  'memoria_lógica_promedio_historias',

  #####Memoria visual
  
  #Tabla 0
  'evocacion_diferida', # -> evocacion_figura_semicompleja
  #Tabla 1
  'caras_codificación',
  'reconocimiento_caras',
  'evocacion_figura_semicompleja',

  #####Genosias | capacidad visuoperceptiva
  #tabla 0
  'imagenes_sobrepuestas', # -> figuras_sobrepuestas
  'matrices',
  #tabla 1
  'imagenes_sobrepuestas',
  

  #####Praxis | praxias
  
  #Tabla 0
  'constructiva', #-> copia_de_figura_compleja
#   'ideomotora_gestos_simbolicos_a_la_orden_derecha',
#   'ideomotora_gestos_simbolicos_a_la_orden_izquierda',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_derecha',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_izquierda',
  #Tabla 1
  'copia_de_figura',
  'gestos_simbolicos',
  'imitacion_de_posturas',

  #####Funciones ejecutivas
  
  #Tabla 0
  'memoria_de_trabajo_digitos_inversos', # -> retencion_digitos_regresion
  'memoria_de_trabajo_digitos_secuenciales', # -> no tiene equivalencia
#   'test_de_fluidez_fonologica_fas_f',
#   'test_de_fluidez_fonologica_fas_a',
#   'test_de_fluidez_fonologica_fas_s',
  'fluidez_verbal_semantica',
  'semejanzas',
  'matrices',
  'comprension_abstraccion', # -> comprension_abstraccion
#   'atencion_alternante',
#   'stroop_a',
#   'stroop_b',
#   'stroop_c',
  'stroop_palabra',
  'stroop_color',
  'stroop_interferencia', # -> stroop_interferencia
  #tabla 1
  'semejanzas', # se puede presentar aqui o en lenguaje
  'fluidez_verbal_semantica',
  'fluidez_verbal_fonologica',
  'fluidez_no_verbal',
  'retencion_digitos_regresion'
  ]

### Mapeo de columas para la tabla con formato 0

In [18]:
features_tabla_0 = [
    
   #Tabla 0 nivel de estudio: Escolaridad baja
   #Tabla 1 nivel de estudio: Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 0
  'atencion_sostenida_auditiva', #'digitos_en_progresion',
  'atencion_sostenida_visual', #no tiene equivalencia
  'atencion_selectiva_visual', #no tiene equivalencia
  'atencion_dividida_visual', #Deteccion visual

  #####Lenguaje
  
  #tabla 0
  'denominacion',
  'material_verbal_complejo', #Es posible que no sea crucial
  'comprension_de_ordenes',

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #verbal del rey 
  
  #Tabla 0 - california verbal learning test
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',
  #tabla 0 - aprendizaje verbal del rey
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',


  #####Memoria visual
  
  #Tabla 0
  'evocacion_diferida',#evocacion_figura_semicompleja

  #####Genosias | capacidad visuoperceptiva
  
  #tabla 0
  'imagenes_sobrepuestas',
  'matrices',

  #####Praxis | praxias
  
  #Tabla 0
  'copia_de_figura', #copia_de_figura_compleja
#   'ideomotora_gestos_simbolicos_a_la_orden_derecha',
#   'ideomotora_gestos_simbolicos_a_la_orden_izquierda',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_derecha',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_izquierda',

  #####Funciones ejecutivas
  
  #Tabla 0
  'memoria_de_trabajo_digitos_inversos', #retencion_digitos_regresion
  'memoria_de_trabajo_digitos_secuenciales', #no tiene equivalencia
#   'test_de_fluidez_fonologica_fas_f',
#   'test_de_fluidez_fonologica_fas_a',
#   'test_de_fluidez_fonologica_fas_s',
  'fluidez_verbal_semantica',
  'semejanzas',
  'matrices',
  'comprension_abstraccion', #comprension_abstraccion
#   'atencion_alternante',
#   'stroop_a',
#   'stroop_b',
#   'stroop_c',
  'stroop_palabra',
  'stroop_color',
  'stroop_interferencia', #stroop_interferencia
  ]

### Mapeo de columnas para la tabla con formato 1

In [19]:
features_tabla_1 = [
    
   #Tabla 0 nivel de estudio: Escolaridad baja
   #Tabla 1 nivel de estudio: Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 1
  'digitos_en_progresion',
#   'cubos_progresion',
  'deteccion_visual',
#   'deteccion_digitos_total',
  'series_sucesivas',

  #####Lenguaje
  

  # #tabla 1
  'denominacion',
  'semejanzas',
  'material_verbal_complejo',
  'comprension_de_ordenes',

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #verbal del rey 
  
  #Tabla 1 #No tiene equivalencia con los otros test de la tabla 0
  'curva_de_memoria_volumen_promedio', 
  'memoria_verbal_espontanea_total',
  'memoria_verbal_claves_total',
  'memoria_verbal_reconocimiento',
  'memoria_logica_promedio_historias',

  #####Memoria visual
  
  #Tabla 1
  'caras_codificacion',
  'reconocimiento_caras',
  'evocacion_figura_semicompleja',

  #####Genosias | capacidad visuoperceptiva
  
  # tabla 1
  'imagenes_sobrepuestas',

  #####Praxis | praxias

  #Tabla 1
  'copia_de_figura',
  'gestos_simbolicos',

  #####Funciones ejecutivas

  #tabla 1
  'semejanzas', # se puede presentar aqui o en lenguaje
  'fluidez_verbal_semantica',
  'fluidez_verbal_fonologica',
  'fluidez_no_verbal',
  'retencion_digitos_regresion',
  ]

### Mapeo de columnas con las dos tablas en conjunto

In [20]:
features_tablas = [
    
   #Tabla 0 nivel de estudio: Escolaridad baja
   #Tabla 1 nivel de estudio: Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 0
  'atencion_sostenida_auditiva', #'digitos_en_progresion',
  'atencion_dividida_visual', #deteccion_visual
  #tabla 1
  'digitos_en_progresion',
  'deteccion_visual',

  #####Lenguaje
  
  #tabla 0
  'denominacion',
  'material_verbal_complejo',
  'semejanzas',
  # #tabla 1
  # 'denominacion'
#   'semejanzas'

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #####Memoria visual
  
  #Tabla 0
  'evocacion_diferida',#evocacion_figura_semicompleja
  #Tabla 1
  'evocacion_figura_semicompleja',

  #####Genosias | capacidad visuoperceptiva
  # Las genosias se quedarian sin analizar

  #####Praxis | praxias
  
  #Tabla 0
  'constructiva', #copia_de_figura_compleja
  #Tabla 1
  'copia_de_figura_compleja',

  #####Funciones ejecutivas
  
  #Tabla 0
  'memoria_de_trabajo_digitos_inversos', #retencion_digitos_regresion
  'comprension_abstraccion', #comprension_abstraccion
  'comprension',
  #tabla 1
  'retencion_digitos_regresion',
  ]

### Hojas procesadas

In [21]:
print(f'Cantidad de datos {len(cleaned_sheets)} \n')
print(f'Nombre de las hojas procesadas:\n ')
sheets = list()
for name, sheet in cleaned_sheets.items():
    sheets.append(name)
print(sheets)

Cantidad de datos 125 

Nombre de las hojas procesadas:
 
['S1-F067-58', 'GC1-60', 'GC2-62', 'F021-72', 'S2-F067-53', 'GC3-60', 'S3-F067-66', 'GC4-72', 'GC5-62', 'S4-F067-66', 'GC6-69', 'GC7-70', 'F022-71', 'GC8-79', 'S5-F067-73', 'F023-87', 'F024-72', 'S6-F067-65', 'F025-82', 'GC8-80', 'GC9-66', 'GC10-65', 'F026-92', 'S7-F067-76', 'S8-F067-75', 'F027-79', 'GC11-72', 'S9-F067-66', 'S10-F067-65', 'GC12-65', 'S11-F067-84', 'F028-70', 's12-f067-77', 'S13-F067-77', 'F029-62', 'S14-F067-69', 'F0210-77', 'F0211-79', 'S15-F067-74', 'F0212-65', 'GC13-74', 'S16-F067-60', 'GC14-67', 'F0213-85', 'S17-F067-76', 'S18-F067-72', 'F0214-73', 'S19-F064-74', 'S20-F067-57', 'S21-F067-68', 'S22-F067-86', 'GC15-64', 'S23-F067-66', 'S24-F067-96', 'GC16-76', 'F0215-73', 'S25-F067-80', 'F0216-83', 'F0217-88', 'F0218-77', 'S26-F067-81', 'S27-F067-66', 'S28-F067-65', 'S29-F067-82', 'F0219-74', 'S30-F067-87', 'S31-F067-83', 'S32-F067-60', 'S33-F067-57', 'F0220-77', 'F0221-57', 'S34-F067-71', 'F0222-64', 'S35-F06

### Formatos de tablas

In [24]:
# Aplicar la detección a todas las hojas de manera vectorizada
type_of_table = {name: detect_table_format(sheet) for name, sheet in cleaned_sheets.items()}
print(f'Cantidad de datos formato tabla 0: {list(type_of_table.values()).count(0)}')
print(f'Cantidad de datos formato tabla 1: {list(type_of_table.values()).count(1)}')
# print(f'Cantidad de datos con formato no determinado: {list(type_of_table.values()).count("no determinada")}')
print(f'Cantidad total de datos procesados: {len(type_of_table)}')

Cantidad de datos formato tabla 0: 36
Cantidad de datos formato tabla 1: 81
Cantidad total de datos procesados: 125


# Creación de los datasets

In [25]:
# Los dos formatos de tablas en conjunto
df_tablas = search_values(cleaned_sheets, features_tablas, type_of_table)

# Formato de tabla 0
cleaned_sheets_tabla_0 = {name: sheet for name, sheet in cleaned_sheets.items() if type_of_table[name] == 0}
df_tabla_0 = search_values(cleaned_sheets_tabla_0, features_tabla_0, type_of_table)

# Formato de tabla 1
cleaned_sheets_tabla_1 = {name: sheet for name, sheet in cleaned_sheets.items() if type_of_table[name] == 1}
df_tabla_1 = search_values(cleaned_sheets_tabla_1, features_tabla_1, type_of_table)

In [26]:
print(f'Grupo de control: {len(df_tablas[df_tablas["dc"] == 0])} personas')
print(f'Grupo de deterioro cognitivo: {len(df_tablas[df_tablas["dc"] == 1])} personas')
print(f'Grupo con demencia: {len(df_tablas[df_tablas["dc"] == 2])} personas')
print(f'Cantidad total de personas en el dataset: {len(df_tablas)} personas')

Grupo de control: 27 personas
Grupo de deterioro cognitivo: 58 personas
Grupo con demencia: 32 personas
Cantidad total de personas en el dataset: 117 personas


### Corrección de nombre y rellenado de nulos

In [30]:
df_tablas['atencion_sostenida_auditiva'] = df_tablas['atencion_sostenida_auditiva'].fillna(df_tablas['digitos_en_progresion'])
df_tablas['atencion_dividida_visual'] = df_tablas['atencion_dividida_visual'].fillna(df_tablas['deteccion_visual'])
df_tablas['evocacion_diferida'] = df_tablas['evocacion_diferida'].fillna(df_tablas['evocacion_figura_semicompleja'])
df_tablas['constructiva'] = df_tablas['constructiva'].fillna(df_tablas['copia_de_figura_compleja'])
df_tablas['memoria_de_trabajo_digitos_inversos'] = df_tablas['memoria_de_trabajo_digitos_inversos'].fillna(df_tablas['retencion_digitos_regresion'])
df_tablas['comprension_abstraccion'] = df_tablas['comprension_abstraccion'].fillna(df_tablas['comprension'])



df_tablas.drop(columns=['digitos_en_progresion'], inplace=True)
df_tablas.drop(columns=['deteccion_visual'], inplace=True)
df_tablas.drop(columns=['evocacion_figura_semicompleja'], inplace=True)
df_tablas.drop(columns=['copia_de_figura_compleja'], inplace=True)
df_tablas.drop(columns=['retencion_digitos_regresion'], inplace=True)
df_tablas.drop(columns=['comprension'], inplace=True)

# atencion_sostenida_auditiva->digitos_en_progresion o
# atencion_dividida_visual->deteccion_visual o
# evocacion_diferida->evocacion_figura_semicompleja o
# constructiva->copia_de_figura_compleja o
# memoria_de_trabajo_digitos_inversos->retencion_digitos_regresion o

# Normalización de datos

In [31]:
# Función para limpiar tildes
def limpiar_tildes(texto):
    texto_nfd = unicodedata.normalize('NFD', str(texto))
    return ''.join(c for c in texto_nfd if unicodedata.category(c) != 'Mn')

def limpiar_caracteres_especiales(texto):
    # Eliminar caracteres especiales excepto espacios y guiones bajos y guiones
    return ''.join(c for c in str(texto) if c.isalnum() or c in [' ', '_', '-'])

def limpiar_texto(texto):
    if pd.isna(texto) or texto == 'None' or texto == 'nan':
        return None
    
    # LUEGO aplicar las limpiezas
    texto = str(texto)
    texto = limpiar_tildes(texto)
    texto = limpiar_caracteres_especiales(texto)
    texto = texto.lower().strip()
    
    return texto if texto else None

### Tabla 0

In [32]:
#Alteracion severa - 1,2,3
#bajo - 4,5,6
#promedio - 7-13
#Alto - 14-18

In [33]:
# Reemplazo de valores
for col in df_tabla_1.columns:
    # Aplicar limpieza de tildes antes de hacer los reemplazos
    df_tabla_1[col] = df_tabla_1[col].apply(limpiar_texto)
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion severa', 'alteracion_severa')
    df_tabla_1[col] = df_tabla_1[col].replace('altearcion severa', 'alteracion_severa')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion leve', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion moderada', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('aleracion leve', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('deficit', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion leve-moderada', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion leve-moderada', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alto', 'alto')
    df_tabla_1[col] = df_tabla_1[col].replace('promedio', 'promedio')
    df_tabla_1[col] = df_tabla_1[col].replace('normal alto', 'alto')
    df_tabla_1[col] = df_tabla_1[col].replace('maximo', 'alto')

In [34]:
#mostrar los posibles valores de todas las columnas menos dc, age, nivel_estudio, sheet_name

cols = df_tabla_1.columns.difference(['dc', 'age', 'nivel_estudio', 'sheet_name'])
for col in cols:
    print(f'Valores únicos en la columna "{col}": {df_tabla_1[col].unique()}')

Valores únicos en la columna "caras_codificacion": ['promedio' None 'alto' 'alteracion_severa']
Valores únicos en la columna "comprension_de_ordenes": ['bajo' None 'alto' 'promedio']
Valores únicos en la columna "copia_de_figura": ['promedio' 'alteracion_severa' 'alto' None 'bajo']
Valores únicos en la columna "curva_de_memoria_volumen_promedio": ['promedio' 'bajo' 'alto' 'alteracion_severa' None]
Valores únicos en la columna "denominacion": ['alto' 'bajo' 'promedio' None]
Valores únicos en la columna "deteccion_visual": ['bajo' 'promedio' 'alteracion_severa' 'alto' None]
Valores únicos en la columna "digitos_en_progresion": ['alteracion_severa' 'promedio' 'bajo' None 'alto']
Valores únicos en la columna "espacio": ['alteracion_severa' 'promedio' None]
Valores únicos en la columna "evocacion_figura_semicompleja": ['promedio' 'bajo' 'alteracion_severa' None 'alto']
Valores únicos en la columna "fluidez_no_verbal": ['promedio' 'bajo' 'alteracion_severa' 'alto' None]
Valores únicos en la 

### Tabla 1

In [35]:
#tabla 1
# menor a 5 alteracion Alteración Severa
# 5 a 24 bajo   
# 25 a 85 promedio
# 85 a 100 Alto

In [36]:
# Reemplazo de valores
for col in df_tabla_0.columns:
    # Aplicar limpieza de tildes antes de hacer los reemplazos
    df_tabla_0[col] = df_tabla_0[col].apply(limpiar_texto)
    df_tabla_0[col] = df_tabla_0[col].replace('alto', 'alto')
    df_tabla_0[col] = df_tabla_0[col].replace('deficit', 'alteracion_severa')
    df_tabla_0[col] = df_tabla_0[col].replace('promedio', 'promedio')
    df_tabla_0[col] = df_tabla_0[col].replace('bajo', 'bajo')
    df_tabla_0[col] = df_tabla_0[col].replace('maximo', 'alto')


In [37]:
#mostrar los posibles valores de todas las columnas menos dc, age, nivel_estudio, sheet_name

cols = df_tabla_0.columns.difference(['dc', 'age', 'nivel_estudio', 'sheet_name'])
for col in cols:
    print(f'Valores únicos en la columna "{col}": {df_tabla_0[col].unique()}')

Valores únicos en la columna "atencion_dividida_visual": [None 'promedio' 'bajo']
Valores únicos en la columna "atencion_selectiva_visual": [None 'promedio' 'bajo']
Valores únicos en la columna "atencion_sostenida_auditiva": [None 'promedio' 'bajo' 'alto']
Valores únicos en la columna "atencion_sostenida_visual": [None 'promedio' 'alteracion_severa' 'bajo' 'alto']
Valores únicos en la columna "comprension_abstraccion": [None 'promedio' 'alteracion_severa' 'alto' 'bajo']
Valores únicos en la columna "comprension_de_ordenes": [None 'alto' 'bajo' 'alteracion_severa']
Valores únicos en la columna "copia_de_figura": [None 'alto' 'alteracion_severa' 'bajo' 'promedio']
Valores únicos en la columna "denominacion": [None 'alto']
Valores únicos en la columna "espacio": [None 'alto' 'bajo']
Valores únicos en la columna "evocacion_diferida": [None 'promedio' 'bajo' 'alto' 'alteracion_severa']
Valores únicos en la columna "evocacion_inmediata_lista_a": [None 'promedio' 'bajo']
Valores únicos en la 

# Verificación de nulos

In [38]:
print(df_tabla_0.shape)
df_tabla_0.isna().sum()

(36, 32)


,0
sheet_name,0
nivel_estudio,0
dc,0
age,0
tiempo,1
persona,1
espacio,1
atencion_sostenida_auditiva,1
atencion_sostenida_visual,4
atencion_selectiva_visual,1


In [39]:
print(df_tabla_1.shape)
df_tabla_1.isna().sum()

(81, 29)


,0
sheet_name,0
nivel_estudio,0
dc,0
age,0
tiempo,3
persona,3
espacio,3
digitos_en_progresion,3
deteccion_visual,6
series_sucesivas,3


# Manejo de valores nulos

In [40]:
# Funciones para perfilado e imputacion clinica de nulos
ID_COLS = ['dc', 'age', 'nivel_estudio', 'sheet_name']


def null_data_info(df, id_cols):
    cols = [c for c in df.columns if c not in id_cols] #Se omiten las columnas de identificación para el perfilado de nulos
    resumen = pd.DataFrame({
        'nulos': df[cols].isna().sum(),
        '%_nulos': (df[cols].isna().mean() * 100).round(2),
        'dtype': df[cols].dtypes.astype(str),
        'n_unicos': df[cols].nunique(dropna=True)
    }).sort_values('%_nulos', ascending=False)
    return resumen


def imputacion_null(df, id_cols, group_col='dc', high_missing=0.30):
    df_imp = df.copy()
    cols = [col for col in df.columns if col not in id_cols]
    missing_rate = df[cols].isna().mean() #El porcentaje de nulos por columna se calcula como el número de nulos dividido por el total de filas, lo que da un valor entre 0 y 1 representando el porcentaje de nulos en cada columna.

    for col in cols:
        porcent = missing_rate[col]
        if porcent == 0:
            continue

        # Indicador de ausencia
        df_imp[f'{col}_missing'] = df[col].isna().astype(int)

        # Si hay demasiados nulos, no imputar
        if porcent >= high_missing:
            continue

        # Imputacion por tipo de dato
        if df[col].dtype == 'object': #Si la columna es de tipo object, se asume que es categórica y se imputa con la moda por grupo clínico. La función fill_mode calcula la moda de cada grupo definido por group_col (en este caso, 'dc') y luego llena los valores nulos con esa moda. Si no hay una moda clara (es decir, si hay un empate), se toma el primer valor de la moda. Si aún quedan nulos después de esta imputación, se realiza una imputación global utilizando la moda de toda la columna.
            # Moda por grupo clinico
            def fill_mode(s):
                #Se calcula la moda de la serie s, omitiendo los valores nulos. Si hay una moda clara, se toma el primer valor; si no, se asigna NaN. Luego, se llena la serie s con este valor de moda para los valores nulos.
                mode = s.mode(dropna=True)
                fill_value = mode.iloc[0] if len(mode) > 0 else np.nan
                return s.fillna(fill_value)

            df_imp[col] = df.groupby(group_col)[col].transform(fill_mode)

            # Fallback global si aun quedan nulos despues de la imputacion por grupo clinico
            if df_imp[col].isna().any():
                global_mode = df[col].mode(dropna=True)
                if len(global_mode) > 0:
                    df_imp[col] = df_imp[col].fillna(global_mode.iloc[0])
        else:
            # Si la variable no es categórica, se asume que es numérica y se imputa con la mediana por grupo clínico. La función fill_median calcula la mediana de cada grupo definido por group_col (en este caso, 'dc') y luego llena los valores nulos con esa mediana. Si aún quedan nulos después de esta imputación, se realiza una imputación global utilizando la mediana de toda la columna.
            df_imp[col] = df.groupby(group_col)[col].transform(lambda s: s.fillna(s.median()))

            # Fallback global si aun quedan nulos
            if df_imp[col].isna().any():
                df_imp[col] = df_imp[col].fillna(df[col].median())

    return df_imp


# Perfil de nulos
print('=== PERFIL NULOS TABLA 0 ===')
perfil_t0 = null_data_info(df_tabla_0, ID_COLS)
print(perfil_t0[perfil_t0['nulos'] > 0])

print('\n=== PERFIL NULOS TABLA 1 ===')
perfil_t1 = null_data_info(df_tabla_1, ID_COLS)
print(perfil_t1[perfil_t1['nulos'] > 0])

# Imputacion clinica (sin eliminar pacientes)
df_tabla_0_imp = imputacion_null(df_tabla_0, ID_COLS)
df_tabla_1_imp = imputacion_null(df_tabla_1, ID_COLS)

print('\nTablas imputadas:')
print('df_tabla_0_imp ->', df_tabla_0_imp.shape)
print('df_tabla_1_imp ->', df_tabla_1_imp.shape)


=== PERFIL NULOS TABLA 0 ===
                                         nulos  %_nulos   dtype  n_unicos
fluidez_verbal_semantica                    23    63.89  object         4
matrices                                    12    33.33  object         4
imagenes_sobrepuestas                       11    30.56  object         3
stroop_interferencia                         9    25.00  object         4
material_verbal_complejo                     6    16.67  object         3
denominacion                                 5    13.89  object         1
stroop_palabra                               4    11.11  object         4
stroop_color                                 4    11.11  object         4
atencion_sostenida_visual                    4    11.11  object         4
evocacion_diferida                           4    11.11  object         4
memoria_de_trabajo_digitos_secuenciales      3     8.33  object         2
comprension_abstraccion                      3     8.33  object         4
semejanza


Tablas imputadas:
df_tabla_0_imp -> (36, 60)
df_tabla_1_imp -> (81, 54)


# Creación de tabla conjunta

In [42]:
# Crear dataframe conjunto por dominios usando puntaje ordinal

def _to_ordinal(series):
    mapping = {
        'alteracion_severa': 1,
        'bajo': 2,
        'promedio': 3,
        'alto': 4
    }
    return series.map(mapping)


def _domain_score(df, cols):
    cols_present = [c for c in cols if c in df.columns]
    if not cols_present:
        return pd.Series([None] * len(df), index=df.index)
    scores = df[cols_present].apply(_to_ordinal)
    return scores.mean(axis=1)


# Unir las dos tablas imputadas
cols_id = ['sheet_name', 'nivel_estudio', 'dc', 'age']
df_union = pd.concat([df_tabla_0_imp, df_tabla_1_imp], ignore_index=True, sort=False)

# Definicion de dominios
dominios = {
    'orientacion': ['tiempo', 'persona', 'espacio'],
    'atencion': [
        'atencion_sostenida_auditiva',
        'atencion_sostenida_visual',
        'atencion_selectiva_visual',
        'atencion_dividida_visual',
        'digitos_en_progresion',
        'deteccion_visual',
        'series_sucesivas'
    ],
    'lenguaje': [
        'denominacion',
        'comprension_de_ordenes',
        'material_verbal_complejo',
        'semejanzas',
        'comprension_ejecucion_de_ordenes'
    ],
    'memoria_verbal': [
        'evocacion_inmediata_lista_a',
        'recuerdo_inmediato_lista_a',
        'recuerdo_inmediato_lista_b',
        'recuerdo_libre_a_corto_plazo',
        'recuerdo_libre_a_largo_plazo',
        'reconocimiento',
        'curva_de_memoria_volumen_promedio',
        'memoria_verbal_espontanea_total',
        'memoria_verbal_claves_total',
        'memoria_verbal_reconocimiento',
        'memoria_logica_promedio_historias'
    ],
    'memoria_visual': [
        'evocacion_diferida',
        'caras_codificacion',
        'reconocimiento_caras',
        'evocacion_figura_semicompleja'
    ],
    'gnosias': ['imagenes_sobrepuestas',
                'matrices'],
    'praxis': [
        'copia_de_figura',
        'gestos_simbolicos',
        'imitacion_de_posturas'
    ],
    'ejecutivas': [
        'memoria_de_trabajo_digitos_inversos',
        'memoria_de_trabajo_digitos_secuenciales',
        'evocacion_categorial_semantica',
        'matrices',
        'comprension_abstraccion',
        'stroop_palabra',
        'stroop_colores',
        'stroop_interferencia',
        'fluidez_verbal_semantica',
        'fluidez_verbal_fonologica',
        'fluidez_no_verbal',
        'retencion_digitos_regresion'
    ]
}

# Construir dataframe por dominios
df_complete = df_union[cols_id].copy()
for dominio, cols in dominios.items():
    df_complete[dominio] = _domain_score(df_union, cols)

In [44]:
df_complete.isnull().sum()

,0
sheet_name,0
nivel_estudio,0
dc,0
age,0
orientacion,0
atencion,0
lenguaje,0
memoria_verbal,0
memoria_visual,0
gnosias,7


In [46]:
# Patron de nulos: resumen, por grupo clinico y co-ocurrencia

def patron_nulos(df, id_cols, group_col='dc'):
    cols = [c for c in df.columns if c not in id_cols]
    miss = df[cols].isna().astype(int)

    print('--- Resumen global ---')
    resumen = miss.mean().sort_values(ascending=False)
    print(resumen[resumen > 0])

    print('\n--- Nulos por grupo clinico (dc) ---')
    grupo = miss.groupby(df[group_col]).mean().sort_index()
    print(grupo)

    print('\n--- Nulos por paciente (distribucion) ---')
    nulos_paciente = miss.sum(axis=1)
    print(nulos_paciente.describe())

    print('\n--- Co-ocurrencia de nulos (top 10 pares) ---')
    corr = miss.corr()
    corr.values[[range(corr.shape[0])], [range(corr.shape[0])]] = 0
    pares = (
        corr.stack()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'level_0': 'var1', 'level_1': 'var2', 0: 'corr_missing'})
    )
    print(pares.head(10))


print('=== PATRON DE NULOS TABLA 0 ===')
patron_nulos(df_tabla_0, ID_COLS)

print('\n=== PATRON DE NULOS TABLA 1 ===')
patron_nulos(df_tabla_1, ID_COLS)


=== PATRON DE NULOS TABLA 0 ===
--- Resumen global ---
fluidez_verbal_semantica                   0.638889
matrices                                   0.333333
imagenes_sobrepuestas                      0.305556
stroop_interferencia                       0.250000
material_verbal_complejo                   0.166667
denominacion                               0.138889
stroop_palabra                             0.111111
stroop_color                               0.111111
atencion_sostenida_visual                  0.111111
evocacion_diferida                         0.111111
memoria_de_trabajo_digitos_secuenciales    0.083333
comprension_abstraccion                    0.083333
semejanzas                                 0.083333
comprension_de_ordenes                     0.055556
copia_de_figura                            0.055556
reconocimiento                             0.055556
recuerdo_inmediato_lista_b                 0.055556
recuerdo_libre_a_corto_plazo               0.055556
recuerdo_

In [47]:
null_data_info(df_complete, ID_COLS)

,nulos,%_nulos,dtype,n_unicos
gnosias,7,5.98,float64,5
orientacion,0,0.00,float64,7
lenguaje,0,0.00,float64,13
atencion,0,0.00,float64,12
memoria_verbal,0,0.00,float64,19
memoria_visual,0,0.00,float64,9
praxis,0,0.00,float64,7
ejecutivas,0,0.00,float64,23


In [48]:
dominios = ['orientacion','atencion','lenguaje','memoria_verbal','memoria_visual','gnosias','praxis','ejecutivas']

#Perfil y patrones por grupo
perfil = df_complete[dominios].isna().mean().sort_values(ascending=False)
print(perfil)

# Imputacion por grupo clinico si nulos moderados
#Se imputan los valores de las praxias por mediana del grupo clinico debido a que el porcenaje es moderado esta por debajo del 1%
for col in dominios:
    if df_complete[col].isna().mean() < 0.30:
        df_complete[col] = df_complete.groupby('dc')[col].transform(lambda s: s.fillna(s.median()))

gnosias           0.059829
orientacion       0.000000
lenguaje          0.000000
atencion          0.000000
memoria_verbal    0.000000
memoria_visual    0.000000
praxis            0.000000
ejecutivas        0.000000
dtype: float64


In [49]:
null_data_info(df_complete, ID_COLS)

,nulos,%_nulos,dtype,n_unicos
orientacion,0,0.0,float64,7
atencion,0,0.0,float64,12
lenguaje,0,0.0,float64,13
memoria_verbal,0,0.0,float64,19
memoria_visual,0,0.0,float64,9
gnosias,0,0.0,float64,5
praxis,0,0.0,float64,7
ejecutivas,0,0.0,float64,23


In [51]:
null_data_info(df_complete, ID_COLS)

,nulos,%_nulos,dtype,n_unicos
orientacion,0,0.0,float64,7
atencion,0,0.0,float64,12
lenguaje,0,0.0,float64,13
memoria_verbal,0,0.0,float64,19
memoria_visual,0,0.0,float64,9
gnosias,0,0.0,float64,5
praxis,0,0.0,float64,7
ejecutivas,0,0.0,float64,23


In [52]:
df_complete.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sheet_name      117 non-null    object 
 1   nivel_estudio   117 non-null    object 
 2   dc              117 non-null    object 
 3   age             117 non-null    object 
 4   orientacion     117 non-null    float64
 5   atencion        117 non-null    float64
 6   lenguaje        117 non-null    float64
 7   memoria_verbal  117 non-null    float64
 8   memoria_visual  117 non-null    float64
 9   gnosias         117 non-null    float64
 10  praxis          117 non-null    float64
 11  ejecutivas      117 non-null    float64
dtypes: float64(8), object(4)
memory usage: 11.1+ KB


In [ ]:
df_complete['nivel_estudio'] = df_complete['nivel_estudio'].astype(str).str.strip().astype(int)
df_complete['age'] = df_complete['age'].astype(str).str.strip().astype(int)
df_complete['dc'] = df_complete['dc'].astype(int)

In [54]:
df_complete["age_num"] = pd.to_numeric(df_complete["age"], errors="coerce")